# 02. Optical-Imaging Datasets and Splits

Model quality cannot repair a scientifically invalid data split. For biomedical imaging, split by the independent biological unit—not by random patches from the same specimen.

## Leakage example

If 100 patches come from one OCT volume and random patches go to both train and test, the model can see nearly identical anatomy/acquisition characteristics in both sets. The test score can become misleadingly optimistic.

In [ ]:
specimens = {
    'train': ['specimen_01', 'specimen_02', 'specimen_03'],
    'val':   ['specimen_04'],
    'test':  ['specimen_05'],
}
assert set(specimens['train']).isdisjoint(specimens['val'])
assert set(specimens['train']).isdisjoint(specimens['test'])
assert set(specimens['val']).isdisjoint(specimens['test'])
specimens

## A Dataset controls how one sample is loaded

Keep preprocessing explicit. Avoid hiding scientifically important operations inside a large transformation chain you cannot audit.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SyntheticOpticalDataset(Dataset):
    def __init__(self, n=32, size=64):
        self.n, self.size = n, size
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        clean = torch.rand(1, self.size, self.size)
        noisy = torch.clamp(clean + 0.08 * torch.randn_like(clean), 0, 1)
        return noisy, clean

loader = DataLoader(SyntheticOpticalDataset(), batch_size=4, shuffle=True)
x, y = next(iter(loader))
print(x.shape, y.shape)

## Before training on real data inspect

- spatial alignment of input/target;
- dtype and intensity range;
- pixel size and resizing;
- channel order;
- whether augmentations are physically plausible;
- specimen IDs after patch extraction.

**Rule:** save a split manifest. A random seed alone is not an adequate record of a clinically or biologically important split.